# 📊 NeMo Fraud Detection: MLflow Evaluation (Zero-Shot & Few-Shot)

This notebook evaluates the base model (**Llama-3.1-8B-Instruct** via NVIDIA NIM) in **Zero-Shot** and **Few-Shot** modes for fraud detection.

### Features:
1. **Mode Control:** Tests the model either with the task instruction only (Zero-Shot) or including positive/negative examples (Few-Shot).
2. **API Communication:** Queries the local NIM endpoint via chat completions.
3. **Metric Calculation & Tracking:** Uses `scikit-learn` for detailed metrics (Accuracy, Precision, Recall, F1-Score) and automatically logs the results to **MLflow**.

In [ ]:
import json
import re
import requests
import mlflow
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# Configuration
VAL_FILE = "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/validation.jsonl"
NIM_URL = "http://172.17.0.1:8800/v1/chat/completions"

# Configure MLflow experiment
mlflow.set_experiment("NeMo-Fraud-Detection-Evaluation")
print("✅ Libraries loaded and MLflow experiment set.")

### 1. Evaluation and Logging Function
This function controls the execution for the respective mode (`zero-shot` or `few-shot`), runs the model, parses the responses, and logs all metrics directly to MLflow.

In [ ]:
def evaluate_and_log(mode="few-shot"):
    print(f"\n🚀 Starting evaluation in mode: {mode.upper()} with MLflow tracking...\n")
    
    y_true = []
    y_pred = []
    total = 0
    passed_judge = 0
    
    with open(VAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            transcript_text = data.get("input", "")
            true_label = data.get("output", "").strip().lower()
            
            if not transcript_text:
                continue
            
            total += 1
            
            # Build messages depending on mode
            messages = [
                {
                    "role": "system",
                    "content": "You are an expert in fraud detection. Briefly analyze the transcript, justify your decision, and end your response with exact 'Result: fraud' or 'Result: legitimate'."
                }
            ]
            
            if mode == "few-shot":
                messages.extend([
                    {
                        "role": "user",
                        "content": "Customer: Hello, I just wanted to ask when my next invoice will be debited. Agent: Good day, the direct debit always takes place on the last day of the month."
                    },
                    {
                        "role": "assistant",
                        "content": "Analysis: Normal billing inquiry with no indicators of fraud. Result: legitimate"
                    },
                    {
                        "role": "user",
                        "content": "Customer: My account has been completely emptied, release the TAN immediately!"
                    },
                    {
                        "role": "assistant",
                        "content": "Analysis: Fear-mongering and pressuring the agent to release a TAN. Classic fraud. Result: fraud"
                    }
                ])
            
            messages.append({"role": "user", "content": transcript_text})
            
            payload = {
                "model": "meta/llama-3.1-8b-instruct",
                "messages": messages,
                "temperature": 0.0,
                "max_tokens": 150
            }
            
            try:
                response = requests.post(NIM_URL, json=payload)
                result = response.json()
                model_output = result["choices"][0]["message"]["content"].strip().lower()
                
                # Parse the result
                if "result: legitimate" in model_output and "fraud" not in model_output:
                    model_answer = "legitimate"
                elif "result: fraud" in model_output and "legitimate" not in model_output:
                    model_answer = "fraud"
                else:
                    match = re.search(r'\b(fraud|legitimate)\b', model_output)
                    model_answer = match.group(1) if match else "unknown"
                
                y_true.append(true_label)
                y_pred.append(model_answer)
                
                if model_answer == true_label:
                    passed_judge += 1
                    
            except Exception as e:
                print(f"Error on entry {total}: {e}")

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=["fraud", "legitimate"], average="weighted", zero_division=0
    )
    judge_rate = (passed_judge / total) * 100 if total > 0 else 0

    print(f"📊 RESULTS FOR {mode.upper()}:")
    print(f"Accuracy: {acc * 100:.2f}% | F1-Score: {f1:.4f} | Match Rate: {judge_rate:.2f}%")
    print("-" * 40)
    print(classification_report(y_true, y_pred, labels=["fraud", "legitimate"], target_names=["fraud", "legitimate"], zero_division=0))

    # Log to MLflow
    with mlflow.start_run(run_name=f"evaluation-{mode}"):
        mlflow.log_param("evaluation_mode", mode)
        mlflow.log_param(
            "model_name", "meta/llama-3.1-8b-instruct"
        )
        mlflow.log_param("dataset_file", VAL_FILE)
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("weighted_f1", f1)
        mlflow.log_metric("weighted_precision", precision)
        mlflow.log_metric("weighted_recall", recall)
        mlflow.log_metric("match_rate_percentage", judge_rate)
        
        print(f"✅ Results successfully logged to MLflow for run mode: {mode}\n")

### 2. Run Experiments (Zero-Shot & Few-Shot)
Both modes are executed sequentially here so that you can compare the results directly in the MLflow UI.

In [ ]:
# Track both modes sequentially
evaluate_and_log(mode="zero-shot")
evaluate_and_log(mode="few-shot")